## Narrative Position

**Pipeline step:** Data lineage (2/6) — 섹션 발췌와 ESG passage 정의

**This notebook answers:** *"어느 섹션을(II/IV/VI) 왜 골랐고, 어떤 문단을 ESG-relevant 하다고 부를 것인가?"*

**Next:** `02_preprocess_kiwi_v2.ipynb`

**Linked robustness evidence:** 섹션 선택 자체가 Decision Box 사안. 본문 셀 안의 *Decision Box* 셀들을 함께 읽을 것.

**Evaluation rubric coverage:** 텍스트 처리 (섹션 선택 정당화)

**Why this matters for our team's framing:** 어디서 텍스트를 잘라내는가가 ESG signal의 *분모*를 결정한다. 섹션 선택은 verbosity와 cheap-talk 해석의 출발 가정이다.

---


# 01 · Section & Passage v2 — ESG 섹션 추출

**Input**  : `data/01_raw/collected_reports.csv`
           + `data/zip_cache/*.zip` (원문 zip)
**Output** : `data/02_sections/extracted_sections_v2.csv`
           + `data/03_passages/esg_passages_v2.parquet`
           + `data/00_meta/passage_filter_log.csv`

---

## 이 단계가 답하는 한 줄

> "사업보고서의 어느 부분에서 ESG 언어를 측정하는가?"

**3개 섹션만 분석하는 이유:**

| 섹션 | 내용 | ESG 관련성 |
|---|---|---|
| II. 사업의 내용 | 주요 사업 설명, 환경/안전 공시 | E/S 공시 중심 |
| IV. 이사의 경영진단 | CEO 메시지, 전략 방향 | E/S/G 통합 |
| VI. 이사회 기관 | 이사회 구성, 감사위원회 | G 공시 중심 |

전체 보고서를 분석하지 않는 이유: III(재무), V(감사의견)는 ESG와 무관한
재무/법무 boilerplate가 지배적이다. 섹션 필터링이 SNR(signal-to-noise ratio)를 높인다.

In [ ]:
import sys, os, re, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline_config import (
    D_META, D_RAW, D_SECTIONS, D_PASSAGES,
    ID_COLS, RANDOM_SEED, SEED_30,
)

SEED_SET = set(SEED_30)
print(f"Seed set size: {len(SEED_SET)}")

## 1. 기존 추출 결과 로드 (재사용)

In [ ]:
# 기존 extracted_sections.csv 있으면 재사용
# 없을 때만 zip_cache에서 재추출
sections_path = D_SECTIONS / 'extracted_sections.csv'

if sections_path.exists():
    sections = pd.read_csv(
        sections_path,
        dtype={'stock_code': str, 'corp_code': str, 'rcept_no': str}
    )
    sections['stock_code'] = sections['stock_code'].astype(str).str.zfill(6)
    print(f'기존 sections 로드: {sections.shape}')
    print(sections['section'].value_counts())
else:
    print('extracted_sections.csv 없음 → 02_extract_sections_fixed.py 실행 필요')
    sections = None

## 2. Sentence Density Filter — ESG 관련 passage 추출

**왜 문단 단위 필터링인가:**  
섹션 전체 텍스트는 수만 자에 달한다.  
ESG seed 단어가 하나도 없는 문단은 noise만 추가한다.  
→ seed 단어 ≥ 1개 포함 문단만 보존 (Passage)

In [ ]:
def extract_passages(
    text: str,
    seed_set: set,
    min_seeds: int = 1,
    min_chars: int = 20,
) -> list:
    """
    text를 문단 단위로 분리, seed 단어 포함 문단만 반환.
    
    Decision:
        Alternative: 문장 단위 필터 (더 세밀)
        Choice: 문단 단위 — 문장 경계 불규칙한 DART XML 특성상
                문단이 더 안정적인 단위.
        Limitation: 문단이 매우 긴 경우 non-ESG 내용이 섞임.
    """
    if not isinstance(text, str) or len(text) < min_chars:
        return []
    
    # 문단 분리 (빈 줄 기준)
    paragraphs = re.split(r'\n{2,}|\r\n{2,}', text)
    
    passages = []
    for para in paragraphs:
        para = para.strip()
        if len(para) < min_chars:
            continue
        
        # 간단한 seed 매칭 (형태소 분석 없이 substring match)
        matched = [s for s in seed_set if s in para]
        
        if len(matched) >= min_seeds:
            passages.append({
                'text':          para,
                'seed_count':    len(matched),
                'matched_seeds': '|'.join(matched[:5]),  # 최대 5개 표시
            })
    
    return passages

# 테스트
test_text = 'ESG 위원회는 탄소중립 목표를 수립했다.\n\n재무제표 주석 참조.\n\n안전보건 관련 임직원 교육을 강화했다.'
result = extract_passages(test_text, SEED_SET)
print(f'Test passages: {len(result)}')
for p in result:
    print(f'  seeds={p["seed_count"]} matched={p["matched_seeds"]}: {p["text"][:60]}')

In [ ]:
import datetime

# 기존 sections가 있으면 passage 재추출
if sections is not None:
    passage_rows = []
    log_rows = []
    
    for (stock_code, corp_code, rcept_no, fiscal_year, esg_year), grp in \
        sections.groupby(['stock_code','corp_code','rcept_no','fiscal_year','esg_year'], observed=False):
        
        doc_passages = []
        for _, sec_row in grp.iterrows():
            sec_name = sec_row['section']
            text     = sec_row.get('text', '')
            
            if not isinstance(text, str) or len(text) < 20:
                continue
            
            passages = extract_passages(text, SEED_SET, min_seeds=1)
            
            for i, p in enumerate(passages):
                passage_rows.append({
                    'stock_code':    stock_code,
                    'corp_code':     corp_code,
                    'rcept_no':      rcept_no,
                    'fiscal_year':   fiscal_year,
                    'esg_year':      esg_year,
                    'section':       sec_name,
                    'paragraph_id':  i,
                    'seed_count':    p['seed_count'],
                    'matched_seeds': p['matched_seeds'],
                    'text':          p['text'],
                })
                doc_passages.append(p)
        
        log_rows.append({
            'stock_code':     stock_code,
            'fiscal_year':    fiscal_year,
            'n_sections':     len(grp),
            'n_passages_out': len(doc_passages),
            'has_passages':   len(doc_passages) > 0,
        })
    
    passages_df = pd.DataFrame(passage_rows)
    log_df = pd.DataFrame(log_rows)
    
    print(f'Passage extraction: {len(passages_df)} passages from {log_df["has_passages"].sum()} firm-year')
    print(f'  no passages: {(~log_df["has_passages"]).sum()} firm-year')
    print(f'  section dist:')
    print(passages_df['section'].value_counts())
    
    # 저장
    passages_path = D_PASSAGES / 'esg_passages_v2.parquet'
    passages_df.to_parquet(passages_path, index=False)
    print(f'→ saved: {passages_path}')
    
    log_path = D_META / 'passage_filter_log.csv'
    log_df.to_csv(log_path, index=False, encoding='utf-8-sig')
    print(f'→ saved: {log_path}')
else:
    print('sections 없음 — passage 추출 스킵')

## 3. firm-year 단위 document 집계 (03_feature_build 입력)

In [ ]:
if 'passages_df' in dir() and len(passages_df) > 0:
    # 섹션별 raw text 합산 (토큰화 이전 raw text document)
    firm_year_docs = passages_df.groupby(
        ['stock_code','corp_code','rcept_no','fiscal_year','esg_year'],
        observed=False
    ).agg(
        n_passages=(  'text', 'count'),
        total_chars=(  'text', lambda x: sum(len(t) for t in x)),
        text=(         'text', lambda x: '\n\n'.join(x.fillna(''))),
        seed_count_sum=('seed_count', 'sum'),
    ).reset_index()
    
    # 섹션별 분리 (II/IV/VI)
    for sec in ['II','IV','VI']:
        sec_df = passages_df[passages_df['section']==sec].groupby(
            ['stock_code','fiscal_year'], observed=False
        ).agg(text_sec=('text', lambda x: ' '.join(x.fillna('')))).reset_index()
        sec_df = sec_df.rename(columns={'text_sec': f'text_{sec}'})
        firm_year_docs = firm_year_docs.merge(
            sec_df, on=['stock_code','fiscal_year'], how='left'
        )
    
    print(f'firm_year_docs: {firm_year_docs.shape}')
    print(f'  avg passages per firm-year: {firm_year_docs["n_passages"].mean():.1f}')
    print(f'  avg chars: {firm_year_docs["total_chars"].mean():.0f}')
    
    # 저장
    fyd_path = D_PASSAGES / 'firm_year_documents_v2.parquet'
    firm_year_docs.to_parquet(fyd_path, index=False)
    print(f'→ saved: {fyd_path}')

## 4. Interpretation

**n_passages 분포 확인:**
- 너무 적으면 (< 3) → ESG 언어가 거의 없는 보고서
- 너무 많으면 (> 100) → seed 단어가 너무 일반적 (boilerplate 의심)

**섹션별 패턴:**
- VI 섹션이 passages 수 최다? → 이사회 섹션에 G 어휘 집중
  → Alpha 3 (Section-Specific Cheap-Talk)의 예비 증거

**다음 단계:** `02_preprocess_kiwi_v2.ipynb`
- firm_year_documents_v2.parquet → Kiwi 토큰화
- 사용자 사전 30개 등록 (NNP, score=50)
- BOILERPLATE_NOUNS 필터링